# GreenTest: JS

This is the **JS** GreenTest notebook. Every language in this repo follows the
same pattern: bootstrap the language, generate a small static site, serve
it locally, and verify it's actually being served correctly.

This one verifies [vanilla-compost](https://github.com/EcologyComputing/vanilla-compost) using Node.js, and walks you through the step-by-step bash commands and concepts.

**Start with python's GreenTest first.** `python/greentest.ipynb` covers the
one-time, per-machine setup every language relies on - git identity and
GitHub authentication - so it isn't repeated here. This notebook's
`bootstrap.sh` still installs its own python3 and Jupyter (in `.venv`
inside this folder), so it runs on its own either way.

Steps:
0. Confirm Node.js is installed
1. Point this notebook at your vanilla-compost clone
2. Confirm it's actually cloned
3. Leave a note about this run
4. Copy `greenTest-Message.md` to vanilla-compost posts folder
5. Generate `posts.html` from the markdown posts using Node.js
6. Serve the site locally using Node.js
7. Verify the generated page is actually being served correctly
8. Clean up

## 0. Confirm Node.js is installed

This bash cell checks that `node` is on your PATH and is v18 or newer, which
is what `bootstrap.sh` installs (via nvm) if it's missing.

In [ ]:
%%bash
if ! which node >/dev/null 2>&1; then
    echo "node was not found on PATH. Run ./bootstrap.sh in this directory,"
    echo "or open a new terminal (so nvm puts node on PATH) and relaunch Jupyter from there."
    exit 1
fi

if ! node -e 'process.exit(parseInt(process.versions.node.split(".")[0]) >= 18 ? 0 : 1)'; then
    echo "Found Node.js $(node --version), but this needs v18 or newer. Re-run ./bootstrap.sh."
    exit 1
fi

echo "Found Node.js $(node --version) at $(which node)"

## 1. Point this notebook at your vanilla-compost clone

GreenTest assumes you cloned vanilla-compost as a sibling of this repo, so
the same folder that has `greenTest/` should also have `vanilla-compost/`.
This cell runs in **bash**. If your clone of vanilla compost lives somewhere else, change the path in the bash script below.

In [1]:
%%bash 
export VANILLA_COMPOST="../../vanilla-compost"
echo "Testing vanilla-compost at: $VANILLA_COMPOST"

Testing vanilla-compost at: ../../vanilla-compost


## 2. Confirm the repo is cloned

This next script is also in bash. It checks that expected files exist and that the folder is a real `git clone`.

In [2]:
%%bash
VANILLA_COMPOST="../../vanilla-compost"
if [ -f "$VANILLA_COMPOST/README.md" ] && [ -f "$VANILLA_COMPOST/src/generate_posts.js" ]; then
    echo "Repo layout looks right (found README.md and src/generate_posts.js)."
else
    echo "That path doesn't look like a vanilla-compost clone with generate_posts.js: $VANILLA_COMPOST"
    exit 1
fi

if git -C "$VANILLA_COMPOST" rev-parse --is-inside-work-tree >/dev/null 2>&1; then
    echo "Confirmed: this is a real git clone, not just a folder of files."
    git -C "$VANILLA_COMPOST" remote get-url origin 2>/dev/null && echo "Its 'origin' remote points there." || echo "No 'origin' remote set."
else
    echo "Warning: no .git found there."
fi

Repo layout looks right (found README.md and src/generate_posts.js).
Confirmed: this is a real git clone, not just a folder of files.
https://github.com/EcologyComputing/vanilla-compost.git
Its 'origin' remote points there.


## 3. Leave a note for this run

Edit the text `"""` marks below with anything about this run. It gets appended to `greenTest-Message.md`.

In [3]:
notes = """
Write your notes, or just "Hello, World!" here before running the rest of the notebook.
"""

In [4]:
from datetime import datetime

log_path = "greenTest-Message.md"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

with open(log_path, "a", encoding="utf-8") as f:
    f.write(f"## {timestamp}\n\n{notes.strip()}\n\n")

print(f"Notes appended to {log_path}")

Notes appended to greenTest-Message.md


## 4. Copy `greenTest-Message.md` to vanilla-compost posts folder

We copy our run log into the blog's `posts/` folder so it will be recognized as a post by the generator script.

In [5]:
import os
import shutil

VANILLA_COMPOST = os.environ.get("VANILLA_COMPOST", "../../vanilla-compost")
dest_dir = os.path.join(VANILLA_COMPOST, "src", "posts")
os.makedirs(dest_dir, exist_ok=True)
shutil.copy("greenTest-Message.md", os.path.join(dest_dir, "greenTest-Message.md"))
print(f"Copied greenTest-Message.md to {dest_dir}/")

Copied greenTest-Message.md to ../../vanilla-compost/src/posts/


## 5. Generate `posts.html`

This cell runs vanilla-compost's `generate_posts.js` using Node.js.

In [6]:
import subprocess

result = subprocess.run(
    ["node", os.path.join(VANILLA_COMPOST, "src", "generate_posts.js")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("generate_posts.js failed - see output above.")

Generated posts.html with 2 posts using html_template.html



Peek at what it generated:

In [7]:
with open(os.path.join(VANILLA_COMPOST, "src", "posts.html"), encoding="utf-8") as f:
    generated = f.read()

start = generated.find('<p class="lead">')
end = generated.find('</p>', start) + len('</p>')
print(generated[start:end] if start != -1 else generated)

<p class="lead">
  <ul>
    <li><a href="post.html?post=greenTest-Message">GreenTest Message</a> <small>(2026-09-23)</small></li>
    <li><a href="post.html?post=hello-compost">Hello Compost</a> <small>(September 23, 2026)</small></li>
  </ul>
</p>


## 6. Serve the site locally

This starts our static file server (`node server.js`) in the background.

In [8]:
import time

server = subprocess.Popen(
    ["node", "server.js", os.path.join(VANILLA_COMPOST, "src")],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)  # give it a moment to start
print(f"Server started (pid {server.pid}) at http://localhost:8080/")

Server started (pid 8695) at http://localhost:8080/


## 7. Verify it's serving the update correctly

We fetch the page from the Node.js server and verify it matches the generated HTML.

In [9]:
import urllib.request

with urllib.request.urlopen("http://localhost:8080/posts.html") as response:
    served = response.read().decode("utf-8")

assert served == generated, "Served posts.html doesn't match what generate_posts.js just wrote."
assert "hello-compost" in served, "Expected the hello-compost post to be listed."

print("Green: the Node.js server is serving the freshly generated posts.html.")

Green: the Node.js server is serving the freshly generated posts.html.


## 8. Clean up

In [10]:
server.terminate()
server.wait()
print("Server stopped.")

Server stopped.
